<a href="https://colab.research.google.com/github/gelisam/colab/blob/main/tara/chapter2_rl/exercises/part22_vpg/2.2.2_VPG_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [2.2.2] - VPG (exercises)

> **ARENA [Course Page](https://learn.arena.education/chapter2_rl/22_vpg/intro/)**
>
> **Colab: [exercises](https://colab.research.google.com/github/ARENA-education/ARENA_materials/blob/main/chapter2_rl/exercises/part22_vpg/2.2.2_VPG_exercises.ipynb?t=20261106) | [solutions](https://colab.research.google.com/github/ARENA-education/ARENA_materials/blob/main/chapter2_rl/exercises/part22_vpg/2.2.2_VPG_solutions.ipynb?t=20261106)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://info-arena.github.io/ARENA_img/slack.html), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://learn.arena.education/chapter0_fundamentals/), [(1) Transformer Interpretability](https://learn.arena.education/chapter1_transformer_interp/), [(2) RL](https://learn.arena.education/chapter2_rl/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-22.png" width="350">

# Introduction

In this section, you'll implement Vanilla Policy Gradient (VPG), the first policy gradient algorithm
upon which more advanced RL algorithms like PPO are based.

## Content & Learning Objectives


### 1️⃣ Policy Gradient Theorem

The Policy Gradient Theorem is what all policy gradient methods are based on:
it allows us to compute the gradient of the return, something that would naively
not have a well defined gradient. You'll then build the policy network.

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem
> - Understand the VPG algorithm: how to perform on-policy policy gradient

### 2️⃣ Rollouts

You'll write a rollout buffer and an agent that steps a batch of GPU CartPole environments with the current policy.

> ##### Learning Objectives
>
> - Collect on-policy experience from many environments in parallel

### 3️⃣ Loss

You'll compute returns, log-probabilities and importance weights, and combine them into the REINFORCE objective with a baseline.

> ##### Learning Objectives
>
> - Turn the policy gradient theorem into a loss you can backpropagate through
> - Understand why a baseline reduces variance without biasing the gradient

### 4️⃣ Training

You'll write the training loop, test it on probe environments, and train Vanilla Policy Gradient (VPG) on CartPole.

> ##### Learning Objectives
>
> - Debug a policy gradient implementation with probe environments
> - Train an agent that balances the pole for the full 500 steps

### Optional Reading

- [Lilian Weng - Policy Gradient Algorithms](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/)
    - A clear and dense overview of the policy gradient theorem and VPG, as well as later algorithms like PPO.

## Setup (don't read, just run!)

In [1]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_materials"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install "wandb>=0.18" einops "gymnasium[atari,other]>=1.2,<2" "ale-py>=0.10" pygame jaxtyping torchinfo git+https://github.com/ARENA-education/eindex.git

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/ARENA-education/ARENA_materials/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

  Cloning https://github.com/ARENA-education/eindex.git to /tmp/pip-req-build-v7j1dx31
  Running command git clone --filter=blob:none --quiet https://github.com/ARENA-education/eindex.git /tmp/pip-req-build-v7j1dx31
  Resolved https://github.com/ARENA-education/eindex.git to commit 525036be2b1a32e77dc5a7dd5f0e6a7bc8a4fdc8
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for eindex-callum: filename=eindex_callum-0.2.0-py3-none-any.whl size=8838 sha256=ef770cc7691f06faa73811ff0ebc41ebaf423407d89d59242e3320625e01c38b
  Stored in directory: /tmp/pip-ephem-wheel-cache-purw54l3/wheels/6e/80/ea/55447ecbe1a1266c6d5328b96578af2268a5564fab2acb2b15
Successfully built eindex-callum
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [eindex-callum]
--2026-11-07 16:35:06--  https://github.com/ARENA-education/ARENA_materials/archive/refs/heads/main.zip
Resolving github.com (github.com)... 140.82.114.3
C

In [12]:
import sys
import time
import warnings
from collections import namedtuple
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Optional

import gymnasium as gym
import numpy as np
import torch as t
import torch.nn.functional as F
import wandb
from eindex import eindex
from gymnasium.spaces import Box, Discrete
from jaxtyping import Bool, Float, Int
from torch import Tensor, nn
from torchinfo import summary
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part22_vpg"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

from gpu_env import CartPole, MountainCar
from gpu_probe import Probe1, Probe2, Probe3, Probe4, Probe5
from rl_utils import ENVS, AtariEnvs, LiveVideo, log_greedy_rollout_video, log_grid_video, make_envs
import part22_vpg.tests as tests
from part1_intro_to_rl.utils import set_global_seeds
from plotly_utils import line, plot_cartpole_obs_and_dones

# The networks in this section are tiny, and if you train on the CPU with
# PyTorch's default of one thread per core, threads spend longer
# coordinating than computing. Adjust if needed.
t.set_num_threads(min(4, t.get_num_threads()))

# CartPole will train just fine on a CPU as we need few parallel environments, and the neural network is very small
# Feel free to change this to "cpu" if you find it runs faster
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


MAIN = __name__ == "__main__"

# 1️⃣ Policy Gradient Theorem

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem
> - Understand the VPG algorithm: how to perform on-policy policy gradient

## Policy Gradient Theorem

Instead of learning action-values and deriving a policy (as in Q-learning or DQN),
**policy gradient methods learn the policy directly**.
- Policy is parameterized: $\pi_\theta(a|s)$ with parameters $\theta$ (often a neural network).  
- Objective: Choose $\theta$ to maximize expected return $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)]$ (joy), where $\tau$ is a trajectory and $G(\tau)$ its return.  

We would desire to update the policy directly via **gradient ascent** against $J(\theta)$:
$$
\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)
$$

The problem is that the return is a sum of rewards from the trajectory, and the trajectory itself is a result of sampling from the policy, over and over,
as well as being dependent on the environmental distribution, which we do not have access to.
There is no clear way to directly compute the gradient of the return with respect to the policy parameters.
The solution here is the **policy gradient theorem**, which states that we can instead use the return weighted by the gradient of the log-probability as an unbiased estimator of the gradient of the return.

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t G_t \nabla_\theta \log \pi_\theta(a_t|s_t) \right]
$$

<details>
<summary>Derivation</summary>

The probability of sampling a trajectory
$\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$
is given by
$$
\Pr(\tau|\theta) = \rho_0(s_0) \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t)
$$
where $\rho_0$ is the initial state distribution and $\mu$ is the environment transition probability.

$$
\begin{align*}
   \nabla_\theta J(\theta) &= \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)] \\
   &= \nabla_\theta \sum_\tau  \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \nabla_\theta \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \Pr(\tau|\theta)\,\nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \\
   &= \mathbb{E}_{\tau \sim \pi_\theta}\left[ \nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \right]
   \end{align*}
   $$
   where we made use of the log-derivative trick: $\nabla_\theta p(x) = p(x) \nabla_\theta \log p(x)$.
  

   The initial state distribution $\rho_0$ and the dynamics $\mu$ do not depend on $\theta$, so:
   $$
   \begin{align*}
   \log \Pr(\tau|\theta) &= \log \left( \rho_0(s_0) \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t) \right) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \log \rho_0(s_0) + \sum_{t=0}^{T-1} \log \mu(s_{t+1}|s_t, a_t) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \text{const.}
   \end{align*}
   $$
   where the const. term is independent of $\theta$, so when we take the gradient, it vanishes.

   Thus:
   $$
   \nabla_\theta \log \Pr(\tau|\theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)
   $$

Plugging back into the gradient:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G(\tau)
\right]
$$

This is the **Vanilla Policy Gradient estimator**, also called **REINFORCE**.
Each $\log \pi_\theta(a_t|s_t)$ is multiplied by the **full return** $G(\tau)$.
However, the action $a_t$ cannot influence rewards before time $t$, only those afterwards.
This means that all the rewards before timestep $t$ merely add noise, as no changes to the policy
can affect them.  To reduce variance, replace $G(\tau)$ with the return $G_t$ at timestep $t$,
also called the **reward-to-go**:
$$
G_t = \sum_{i=t}^{T} \gamma^{i-t} r_{i}
$$

Thus, the lower-variance unbiased estimator is:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G_t
\right]
$$

</details>

There are many other variants of the policy gradient estimator, as described in [Schulman et al., 2015](https://arxiv.org/abs/1506.02438).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/policy_grad.png" width="800">

## Implementation

We make use of exactly the same vectorised GPU `CartPole` environment as we did for DQN (see `chapter2_rl/exercises/gpu_env.py`): it's defined entirely in terms of tensor operations, so

- we don't need to constantly convert between numpy and torch tensors
- we can run large numbers of environments in parallel (thousands of environments, for millions of environment steps per second)
- we avoid copying data back and forth between the CPU and GPU, which can be a significant bottleneck

For DQN we only ran a handful of environments at once, because DQN's sample efficiency comes from the replay buffer rather than from how much fresh data we collect per step. (Vanilla) Policy gradient methods can't reuse old data in the same way, so here we'll lean on the vectorisation much harder and collect a rollout from *thousands* of environments at once every update.

## The Policy Network

The Policy Network takes in an observation $s$ and outputs a vector of logits, whose softmax $\pi(a_1 | s), \ldots \pi(a_n | s)$ gives the probability
that the network takes action $a_i$ given observation $s$. We will then adjust the parameters of this network to directly maximize the expected return.

For learning from pixels, a simple convolutional network and some fully connected layers does quite well. Where we have already processed features here, it's even easier: a small MLP like this one is plenty for CartPole and the other classic-control environments.

Implement the Policy Network using a standard MLP, constructed of alternating Linear and ReLU layers.
The size of the input will match the dimensionality of the observation space, and the size of the output will match the number of actions to choose from (associating a logit with each action).
The dimensions of the hidden_sizes are provided.

Here is a diagram of what our particular Policy Network will look like for CartPole (you can open it in a new tab if it's hard to see clearly):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ch2-qnetwork-mlp-obs-dim.svg" style="background-color: #fffbe0; padding: 10px" width="160"><br>

<!--
flowchart TD
    A["Input (obs_dim,)"] -> B["Linear(obs_dim, 120)"]
    B -> C[ReLU]
    C -> D["Linear(120, 84)"]
    D -> E[ReLU]
    E -> F["Linear(84, num_actions)"]
    F -> G["Output (num_actions,)"]
{
  "theme": "default",
  "themeVariables": {
    "fontSize": "22px"
  }
}
-->

<details>
<summary>Question - why do we not include a ReLU at the end?</summary>

If you end with a ReLU, then the logits are capped below at 0, which means the
network can only push down the undesirability of an action so far. We want
the output to be unbounded so the network is free to disprefer an action as much as it wants.
</details>

## My answer 😐

Because drops information without providing useful non-linearity. We should let the network say that a particular action has negative value, we don't need to restrict the output range because softmax is already doing that.

### Exercise - implement `PolicyNetwork`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Note - in this implementation we can assume that `obs_shape` is a tuple of length 1 (in the case of CartPole this will be `(4,)`), so you can treat it as just an integer value above, e.g. your first linear layer should be from `obs_shape[0]` to `120`.

In [4]:
class PolicyNetwork(nn.Module):
    """
    For consistency with our tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()
        assert len(obs_shape) == 1, "Expecting a single vector of observations"
        self.layers = nn.Sequential(
            nn.Linear(obs_shape[0], hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], num_actions)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


net = PolicyNetwork(obs_shape=(4,), num_actions=2)
n_params = sum((p.nelement() for p in net.parameters()))
print(net)
print(f"Total number of parameters: {n_params}")
tests.test_policy_network(PolicyNetwork)

PolicyNetwork(
  (layers): Sequential(
    (0): Linear(in_features=4, out_features=120, bias=True)
    (1): ReLU()
    (2): Linear(in_features=120, out_features=84, bias=True)
    (3): ReLU()
    (4): Linear(in_features=84, out_features=2, bias=True)
  )
)
Total number of parameters: 10934
All tests in `test_policy_network` passed!


<details><summary>Solution</summary>

```python
class PolicyNetwork(nn.Module):
    """
    For consistency with your tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()
        assert len(obs_shape) == 1, "Expecting a single vector of observations"
        in_features_list = [obs_shape[0]] + hidden_sizes
        out_features_list = hidden_sizes + [num_actions]
        layers = []
        for i, (in_features, out_features) in enumerate(zip(in_features_list, out_features_list)):
            layers.append(nn.Linear(in_features, out_features))
            if i < len(in_features_list) - 1:
                layers.append(nn.ReLU())
        self.layers = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)
```
</details>

# 2️⃣ Rollouts

> ##### Learning Objectives
>
> - Collect on-policy experience from many environments in parallel

## Rollout Buffer

The way that our implementation of VPG will work is simple: we perform a rollout across `num_envs` many environments in parallel, and store the trajectories for each. We then learn from that set of rollouts, and then discard it afterwards. One rollout, one learning step. This means we are always learning **on-policy**: we only ever learn from data that the current model actually generated. We will use a rollout buffer to store the trajectories.

### Exercise - implement `Rollout Buffer`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 20 minutes on this exercise.
> ```

The `Rollout` class will store a set of `num_envs` many trajectories. We do not shuffle up anything, or break up an episode into little experiences as we did for DQN. The smallest datapoint is one full trajectory:

$$\tau = s_0 \; a_0 \; r_0 \; s_1 \; a_1 \; r_1 \ldots s_T \; a_T \; r_T$$

The following methods need to be completed:

* `add_step` - adds information gathered from timestep $t$ to the rollout buffer
* `get_batches` - returns a list of `RolloutTensors` objects, each containing `batch_size` many trajectories.

We store the tensors for each step as separate lists, and then stack once at the end to get the final tensors with the `.get` function. This ends up being cheaper as it avoids spinning up indexed-write kernels per step.

<details>
<summary>Hint</summary>

Use `t.split` to write `get_batches`.
</details>

In [39]:
import einops
RolloutTensors = namedtuple("RolloutTensors", ["obs", "actions", "logprobs", "rewards", "dones"])


class Rollout:
    _obs: list[Float[Tensor, " num_envs *obs_shape"]]
    _actions: list[Int[Tensor, " num_envs"]]
    _logprobs: list[Float[Tensor, " num_envs"]]
    _rewards: list[Float[Tensor, " num_envs"]]
    _dones: list[Bool[Tensor, " num_envs"]]
    timestep: int

    def __init__(self, max_steps: int):
        """
        Args:
            max_steps: maximum number of steps to rollout per environment (the per-step tensors carry the
                       number of environments and the observation/action shapes themselves)
        """

        self.MAX_SIZE = max_steps

        # Per-step we append tensor references to Python lists (free) and t.stack() once at the
        # end, instead of 5 indexed-write kernels per step into a preallocated buffer. Each stored
        # tensor is freshly produced per step (the env returns a new state tensor each step), so
        # holding references is safe. This removes ~2500 tiny kernel launches per full rollout.
        self._obs, self._actions, self._logprobs, self._rewards, self._dones = [], [], [], [], []
        self.timestep = 0

    def add_step(
        self,
        obs: Float[Tensor, " num_envs *obs_shape"],
        actions: Int[Tensor, " num_envs"],
        logprobs: Float[Tensor, " num_envs"],
        rewards: Float[Tensor, " num_envs"],
        dones: Bool[Tensor, " num_envs"],
    ):
        """
        Stores one timestep of the rollout (the same transition for all `num_envs` environments at once)
        and advances `self.timestep`.

        Args:
            obs:      (num_envs, *obs_shape)  the observation (o_t) the agent saw *before* acting
            actions:  (num_envs,)             the action (a_t) it took in each env after seeing o_t
            logprobs: (num_envs,)             log pi(a_t | o_t) of the action that was *taken*
                                              Recorded at collection time, so
                                              these are the behaviour policy's logprobs (used for importance weights)
            rewards:  (num_envs,)             the reward (r_{t+1}) received for taking a_t
            dones:    (num_envs,)  bool       True if that env's episode ended on this step (d_{t+1} = terminated OR truncated),
                                              i.e. the env auto-reset and the *next* stored obs starts a new episode

        Raises ValueError if the rollout already holds `max_steps` timesteps.
        """

        if self.timestep >= self.MAX_SIZE:
            raise ValueError("Rollout is full, cannot add more steps")

        self._obs.append(obs)
        self._actions.append(actions)
        self._logprobs.append(logprobs)
        self._rewards.append(rewards)
        self._dones.append(dones)
        self.timestep += 1
        # is that it??

    def reset(self):
        self._obs.clear(); self._actions.clear(); self._logprobs.clear()
        self._rewards.clear(); self._dones.clear()
        self.timestep = 0

    def get(self) -> tuple[Tensor, ...]:
        """
        Stack the per-step lists from (num_envs, *tensor_shape) into (num_envs, timestep, *tensor_shape) tensors. Rollouts can stop early
        (see gen_rollout), so the time dimension is however many steps were actually collected.
        """
        assert self.timestep > 0, "Rollout is empty"
        return RolloutTensors(
            t.stack(self._obs, dim=1),             # num_envs max_steps obs_size
            t.stack(self._actions, dim=1),         # num_envs max_steps
            t.stack(self._logprobs, dim=1),        # num_envs max_steps
            t.stack(self._rewards, dim=1).float(), # num_envs max_steps
            t.stack(self._dones, dim=1),           # num_envs max_steps
        )

    def get_batches(self, batch_size: int) -> list[RolloutTensors]:
        """
        Splits the rollout buffer into batches of size `batch_size`, and returns a list of
        `RolloutTensors` objects, each containing `batch_size` many trajectories.
        """
        # we are returning a list of _RolloutTensors_, not a list of
        # trajectories, so we don't need to split the data at the timesteps
        # where _done[t] is True?

        obs      = t.stack(self._obs, dim=1)             # num_envs max_steps obs_size
        actions  = t.stack(self._actions, dim=1)         # num_envs max_steps
        logprobs = t.stack(self._logprobs, dim=1)        # num_envs max_steps
        rewards  = t.stack(self._rewards, dim=1).float() # num_envs max_steps
        dones    = t.stack(self._dones, dim=1)           # num_envs max_steps

        obs      = einops.rearrange(obs,      "(n batch_size) max_steps obs_size -> n batch_size max_steps obs_size", batch_size=batch_size)
        actions  = einops.rearrange(actions,  "(n batch_size) max_steps -> n batch_size max_steps", batch_size=batch_size)
        logprobs = einops.rearrange(logprobs, "(n batch_size) max_steps -> n batch_size max_steps", batch_size=batch_size)
        rewards  = einops.rearrange(rewards,  "(n batch_size) max_steps -> n batch_size max_steps", batch_size=batch_size)
        dones    = einops.rearrange(dones,    "(n batch_size) max_steps -> n batch_size max_steps", batch_size=batch_size)

        r = []
        for i in range(len(dones)):
          r.append(RolloutTensors(
              obs[i],
              actions[i],
              logprobs[i],
              rewards[i],
              dones[i],
          ))
        return r


        # r = []
        # while self.timestep >= batch_size:
        #   r.append(
        #       RolloutTensors(
        #           t.stack(self._obs[0:batch_size], dim=1),             # num_envs batch_size obs_size
        #           t.stack(self._actions[0:batch_size], dim=1),         # num_envs batch_size
        #           t.stack(self._logprobs[0:batch_size], dim=1),        # num_envs batch_size
        #           t.stack(self._rewards[0:batch_size], dim=1).float(), # num_envs batch_size
        #           t.stack(self._dones[0:batch_size], dim=1),           # num_envs batch_size
        #       )
        #   )
        #   self._obs      = self._obs[batch_size:]
        #   self._actions  = self._actions[batch_size:]
        #   self._logprobs = self._logprobs[batch_size:]
        #   self._rewards  = self._rewards[batch_size:]
        #   self._dones    = self._dones[batch_size:]
        #   self.timestep -= batch_size
        # return r

values = t.tensor([1,2,3,4,5,6,7,8])
dones = t.tensor([False, False, True, False, True, False, False, True])
# want to split into [[1,2,3],[4,5],[6,7,8]]
indices = t.arange(len(dones))
end_indices = t.cat([t.tensor([-1]), indices[dones]])
sizes = end_indices[1:] - end_indices[:-1]
t.split(values, tuple(sizes))

tests.test_rollout(Rollout)

All tests in `test_rollout` passed!


<details><summary>Solution</summary>

```python
RolloutTensors = namedtuple("RolloutTensors", ["obs", "actions", "logprobs", "rewards", "dones"])


class Rollout:
    _obs: list[Float[Tensor, " num_envs *obs_shape"]]
    _actions: list[Int[Tensor, " num_envs"]]
    _logprobs: list[Float[Tensor, " num_envs"]]
    _rewards: list[Float[Tensor, " num_envs"]]
    _dones: list[Bool[Tensor, " num_envs"]]
    timestep: int

    def __init__(self, max_steps: int):
        """
        Args:
            max_steps: maximum number of steps to rollout per environment (the per-step tensors carry the
                       number of environments and the observation/action shapes themselves)
        """
        
        self.MAX_SIZE = max_steps

        # Per-step we append tensor references to Python lists (free) and t.stack() once at the
        # end, instead of 5 indexed-write kernels per step into a preallocated buffer. Each stored
        # tensor is freshly produced per step (the env returns a new state tensor each step), so
        # holding references is safe. This removes ~2500 tiny kernel launches per full rollout.
        self._obs, self._actions, self._logprobs, self._rewards, self._dones = [], [], [], [], []
        self.timestep = 0

    def add_step(
        self,
        obs: Float[Tensor, " num_envs *obs_shape"],
        actions: Int[Tensor, " num_envs"],
        logprobs: Float[Tensor, " num_envs"],
        rewards: Float[Tensor, " num_envs"],
        dones: Bool[Tensor, " num_envs"],
    ):
        """
        Stores one timestep of the rollout (the same transition for all `num_envs` environments at once)
        and advances `self.timestep`.

        Args:
            obs:      (num_envs, *obs_shape)  the observation (o_t) the agent saw *before* acting
            actions:  (num_envs,)             the action (a_t) it took in each env after seeing o_t
            logprobs: (num_envs,)             log pi(a_t | o_t) of the action that was *taken*
                                              Recorded at collection time, so
                                              these are the behaviour policy's logprobs (used for importance weights)
            rewards:  (num_envs,)             the reward (r_{t+1}) received for taking a_t
            dones:    (num_envs,)  bool       True if that env's episode ended on this step (d_{t+1} = terminated OR truncated),
                                              i.e. the env auto-reset and the *next* stored obs starts a new episode

        Raises ValueError if the rollout already holds `max_steps` timesteps.
        """

        if self.timestep >= self.MAX_SIZE:
            raise ValueError("Rollout is full, cannot add more steps")

        self._obs.append(obs)
        self._actions.append(actions)
        self._logprobs.append(logprobs)
        self._rewards.append(rewards)
        self._dones.append(dones)
        self.timestep += 1

    def reset(self):
        self._obs.clear(); self._actions.clear(); self._logprobs.clear()
        self._rewards.clear(); self._dones.clear()
        self.timestep = 0

    def get(self) -> tuple[Tensor, ...]:
        """
        Stack the per-step lists from (num_envs, *tensor_shape) into (num_envs, timestep, *tensor_shape) tensors. Rollouts can stop early
        (see gen_rollout), so the time dimension is however many steps were actually collected.
        """
        assert self.timestep > 0, "Rollout is empty"
        return RolloutTensors(
            t.stack(self._obs, dim=1),
            t.stack(self._actions, dim=1),
            t.stack(self._logprobs, dim=1),
            t.stack(self._rewards, dim=1).float(),
            t.stack(self._dones, dim=1),
        )

    def get_batches(self, batch_size: int) -> list[RolloutTensors]:
        """
        Splits the rollout buffer into batches of size `batch_size`, and returns a list of
        `RolloutTensors` objects, each containing `batch_size` many trajectories.
        """

        tau = self.get()  # filled portion only
        obs = t.split(tau.obs, batch_size, dim=0)
        acts = t.split(tau.actions, batch_size, dim=0)
        logprobs = t.split(tau.logprobs, batch_size, dim=0)
        rewards = t.split(tau.rewards, batch_size, dim=0)
        dones = t.split(tau.dones, batch_size, dim=0)

        batches = [RolloutTensors(*tensors) for tensors in zip(obs, acts, logprobs, rewards, dones)]

        return batches


tests.test_rollout(Rollout)
```
</details>

## VPG Args

We've provided a dataclass for the training arguments, and will explain as needed later on.

In [40]:
@dataclass
class VPGArgs:
    # Basic / global
    seed: int = 1
    env_id: str = "CartPole-gpu"

    # Wandb / logging
    use_wandb: bool = False
    wandb_project_name: str = "VPGCartPole"
    wandb_entity: str | None = None
    video_log_freq: int | None = 50   # every N gradient steps, render a 4x4 grid video of the latest rollout
                                      # (logged to wandb if use_wandb, shown inline if live_viz)

    # Duration of different phases / buffer memory settings
    total_timesteps: int = 500_000
    # max_rollout_steps: int = 500
    # min_rollout_steps: int = 64
    num_envs: int = 4

    num_steps_per_rollout: int = 128

    lr: float = 2.5e-4
    gamma: float = 1
    rollout_use_count: int = 1
    clip_coef: float = 0.2
    device: str = "cpu"
    normalize_returns: bool = True
    num_batches_per_rollout: int = 1
    # LR decay settings
    use_lr_decay: bool = False
    lr_end: Optional[float] = None
    lr_frac: Optional[float] = None
    use_iw: bool = False
    full_reset: bool = True   # fully reset all envs at the start of each rollout
    live_viz: bool = True     # display the 4x4 grid video inline every `video_log_freq` gradient steps (notebook only)
    overwrite_video: bool = True  # each live video replaces the previous one in the notebook output (False appends them)

    def __post_init__(self):
        self.batch_size = self.num_envs // self.num_batches_per_rollout
        self.device = t.device(self.device)

        if self.use_lr_decay:
            assert self.lr_end is not None, "lr_end must be set if use_lr_decay is True"
            assert self.lr_frac is not None, "lr_frac must be set if use_lr_decay is True"

        self.env_steps_per_update = self.num_steps_per_rollout * self.num_envs // self.num_batches_per_rollout

        if not self.use_iw:
            assert self.rollout_use_count == 1, "rollout_use_count must be 1 if use_iw is False"
            assert self.num_batches_per_rollout == 1, "num_batches_per_rollout must be 1 if use_iw is False"

## VPG Agent

The following class will be our agent, that will generate rollouts via interaction between the agent and environment, as well as generate actions by sampling them from the policy network. Recall that the policy network now maps observations to logits for each action, so we can sample actions from the distribution.

### Exercise - implement `VPGAgent`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Implement the functions:
* `gen_rollout` - this function computes the episode rollout, by interacting with the environment for `args.num_steps_per_rollout` steps. If an episode terminates, the environment resets itself and we keep going, so a rollout usually contains several episodes per environment.

* `get_actions` - this function takes in an observation, and returns the actions and their logprobs. Sample the actions with `t.multinomial` on the softmax of the policy network's logits (see [the docs](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)), then gather the logprob of each sampled action.

To see how well the agent is doing we don't need any extra bookkeeping during the rollout: afterwards, the training loop sums each environment's rewards up to (and including) its first `done` to get the undiscounted return of its first episode, and reports the mean over environments with its standard error. For CartPole (+1 per step alive, max 500) that number is the episode length.

In [63]:
class VPGAgent:
    """Base Agent class handling the interaction with the environment."""

    def __init__(
        self,
        envs: gym.Env,
        policy_network: PolicyNetwork,
        args: VPGArgs,
    ):

        self.envs = envs
        self.policy_network = policy_network
        self.args = args
        self.obs_shape = envs.observation_space.shape
        self.action_shape = envs.action_space.shape

    @t.no_grad()
    def gen_rollout(self, rollout: Rollout) -> Rollout:
        """
        Steps all environments in parallel for `args.num_steps_per_rollout` steps, adding each step to the rollout
        buffer (which is reset first), and returns the filled buffer.
        """
        if self.args.full_reset and hasattr(self.envs, "_terminated"):
            self.envs._terminated[:] = True
            self.envs._truncated[:] = True
        obs, _ = self.envs.reset()  # Need a starting observation
        rollout.reset()

        for i in range(self.args.num_steps_per_rollout):
          actions, logprobs = self.get_actions(obs)
          (new_obs, rewards, terminated, truncated, _info) = self.envs.step(actions)
          #print(terminated[:3])
          #print(truncated[:3])
          dones = t.logical_or(terminated, truncated)
          rollout.add_step(obs, actions, logprobs, rewards, dones)
          obs = new_obs

        return rollout

    def get_actions(
        self, obs: Float[Tensor, " num_envs *obs_shape"]
    ) -> tuple[Int[Tensor, " num_envs"], Float[Tensor, " num_envs"]]:
        """
        Computes the agent's turn: given an observation for each environment,
        sample the action the agent takes, along with the log_probs of that action.
        Use t.multinomial to sample the actions.
        """
        #print(obs[:3])
        logits = self.policy_network.forward(obs)  # num_envs actions
        #print(logits[:3])
        logprobs = t.log_softmax(logits, dim=1)  # num_envs actions
        #print(logprobs[:3])
        probs = t.exp(logprobs)  # num_envs actions
        #print(probs[:3])
        actions = t.multinomial(probs, 1)  # num_envs 1
        #print(actions[:3])
        sampled_logprobs = logprobs.gather(index=actions, dim=1)  # num_envs 1
        #print(sampled_logprobs[:3])
        return (
            einops.rearrange(actions, "num_envs 1 -> num_envs"),
            einops.rearrange(sampled_logprobs, "num_envs 1 -> num_envs")
        )

tests.test_get_actions(VPGAgent, PolicyNetwork)
tests.test_gen_rollout(VPGAgent, PolicyNetwork, VPGArgs, Rollout)

All tests in `test_get_actions` passed!
All tests in `test_gen_rollout` passed!


<details><summary>Solution</summary>

```python
class VPGAgent:
    """Base Agent class handling the interaction with the environment."""

    def __init__(
        self,
        envs: gym.Env,
        policy_network: PolicyNetwork,
        args: VPGArgs,
    ):

        self.envs = envs
        self.policy_network = policy_network
        self.args = args
        self.obs_shape = envs.observation_space.shape
        self.action_shape = envs.action_space.shape

    @t.no_grad()
    def gen_rollout(self, rollout: Rollout) -> Rollout:
        """
        Steps all environments in parallel for `args.num_steps_per_rollout` steps, adding each step to the rollout
        buffer (which is reset first), and returns the filled buffer.
        """
        if self.args.full_reset and hasattr(self.envs, "_terminated"):
            self.envs._terminated[:] = True
            self.envs._truncated[:] = True
        obs, _ = self.envs.reset()  # Need a starting observation
        rollout.reset()

        for timestep in range(self.args.num_steps_per_rollout):
            actions, logprobs = self.get_actions(obs)
            new_obs, rewards, terminates, truncates, info = self.envs.step(actions)
            # Mask returns at episode boundaries on EITHER termination or truncation: the env
            # auto-resets on both, so returns must not sum across the reset boundary.
            done = terminates | truncates
            rollout.add_step(obs, actions, logprobs, rewards, done)
            obs = new_obs

        return rollout

    def get_actions(
        self, obs: Float[Tensor, " num_envs *obs_shape"]
    ) -> tuple[Int[Tensor, " num_envs"], Float[Tensor, " num_envs"]]:
        """
        Computes the agent's turn: given an observation for each environment,
        sample the action the agent takes, along with the log_probs of that action.
        Use t.multinomial to sample the actions.
        """
        logits = self.policy_network(obs)  # (num_envs, num_actions) unnormalised scores
        log_probs = F.log_softmax(logits, dim=-1)  # log pi(a|s): logits - logsumexp, numerically stable & differentiable
        # One action index per env, drawn with probability pi(a|s) (multinomial renormalises each row, so we just need
        # non-negative weights). The result is an integer tensor: no gradient flows through a discrete draw, and none should.
        actions = t.multinomial(log_probs.exp(), num_samples=1).squeeze(-1)
        # alternative with eindex
        # logprobs_per_act = eindex(log_probs, actions, "env [env] -> env")
        # alternative with gather
        logprobs_per_act = log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)        
        return actions, logprobs_per_act


tests.test_get_actions(VPGAgent, PolicyNetwork)
tests.test_gen_rollout(VPGAgent, PolicyNetwork, VPGArgs, Rollout)
```
</details>

# 3️⃣ Loss

> ##### Learning Objectives
>
> - Turn the policy gradient theorem into a loss you can backpropagate through
> - Understand why a baseline reduces variance without biasing the gradient

## Returns

To compute the REINFORCE loss, we need to compute the return for each step in the trajectory. This gets a little messy as trajectories may be of different lengths, so an episode may have terminated part way through the rollout. You'll need to walk backward through the trajectory, and compute the return for each step.

Note that `gen_rollout` stores `done = terminated | truncated`. This is the same `done` that the DQN section computed when it stepped its environments: it marks an *episode boundary*.

A real truncated tail does remain, but it comes from the rollout window rather than from either flag. `gen_rollout` runs for `num_steps_per_rollout` steps and your backward pass will start from $G = 0$, so an environment which is *still alive* when the window ends gets no credit at all for the reward it would have gone on to collect. The actor-critic methods in the PPO section fix this: with a critic you can bootstrap $V(s_T)$ at the end of the window instead of pretending the future is worth zero.

### Exercise - implement `compute_returns`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Compute the returns for each trajectory. Easiest to write as a simple reverse for-loop for now, though if you wish later on you can try a vectorized solution.

In [72]:
def compute_returns(
    rewards: Float[Tensor, " num_envs num_steps"], dones: Bool[Tensor, " num_envs num_steps"], gamma: float = 0.9
):
    """
    ARGS:
        rewards: The rewards for each trajectory
        done: A boolean tensor indicating if an episode finished on the current timestep
        gamma: The discount factor

    Returns:
        The returns G_t for each trajectory.

        For example:
        - If Rewards = [0, 0, 1, 0, 1]
        - And Done   = [0, 0, 1, 0, 1]
        - Then Returns = [g**2, g, 1, g, 1]
    """
    num_envs, num_steps = rewards.shape

    returns = t.zeros_like(rewards)

    for i in range(num_envs):
      total_reward = 0.0
      for j in range(num_steps-1, -1, -1):
        reward = rewards[i][j]
        done = dones[i][j]
        if done:
          total_reward = reward
        else:
          total_reward = gamma * total_reward + reward
        returns[i][j] = total_reward
    return returns

print(compute_returns(
    t.tensor([[0.0, 0.0, 1.0, 0.0, 1.0]]),
    t.tensor([[0.0, 0.0, 1.0, 0.0, 1.0]]),
    0.5,
))

tests.test_compute_returns(compute_returns)

tensor([[0.2500, 0.5000, 1.0000, 0.5000, 1.0000]])
All tests in `test_compute_returns` passed!


<details><summary>Solution</summary>

```python
def compute_returns(
    rewards: Float[Tensor, " num_envs num_steps"], done: Bool[Tensor, " num_envs num_steps"], gamma: float = 0.9
):
    """
    ARGS:
        rewards: The rewards for each trajectory
        done: A boolean tensor indicating if an episode finished on the current timestep
        gamma: The discount factor

    Returns:
        The returns G_t for each trajectory.

        For example:
        - If Rewards = [0, 0, 1, 0, 1]
        - And Done   = [0, 0, 1, 0, 1]
        - Then Returns = [g**2, g, 1, g, 1]
    """
    num_envs, num_steps = rewards.shape

    returns = t.zeros_like(rewards)


    G = t.zeros_like(rewards[:, 0])  # (num_envs)
    for i in reversed(range(num_steps)):
        G = rewards[:, i] + gamma * G * (~done[:, i])
        returns[:, i] = G
    return returns


tests.test_compute_returns(compute_returns)
```
</details>

### Exercise - implement `compute_logprobs`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Computes the log-probability under the *current* policy of the action that was taken on each timestep. To pick out the logprob of each taken action from the `(num_envs, num_steps, num_actions)` log-probs, use `gather` along the last dimension (as `get_actions` does), or the eindex library.

In [79]:
def compute_logprobs(tau: RolloutTensors, pi: PolicyNetwork) -> Float[Tensor, " num_envs num_steps"]:
    """
    Computes the logprobs of the actions taken on each timestep under the current policy.
    """
    obs = tau.obs  # num_envs max_steps obs_size
    logits = pi.forward(tau.obs)  # num_envs max_steps actions
    logprobs = t.log_softmax(logits, dim=2)  # num_envs max_steps actions
    probs = t.exp(logprobs)  # num_envs max_steps actions
    actions = einops.repeat(tau.actions, "num_envs max_steps -> num_envs max_steps 1")
    sampled_logprobs = logprobs.gather(index=actions, dim=2)  # num_envs max_steps 1
    return einops.rearrange(sampled_logprobs, "num_envs max_steps 1 -> num_envs max_steps")


tests.test_compute_logprobs(compute_logprobs, PolicyNetwork)

All tests in `test_compute_logprobs` passed!


<details><summary>Solution</summary>

```python
def compute_logprobs(tau: RolloutTensors, pi: PolicyNetwork) -> Float[Tensor, " num_envs num_steps"]:
    """
    Computes the logprobs of the actions taken on each timestep under the current policy.
    """
    logits = pi(tau.obs)
    log_probs = F.log_softmax(logits, dim=-1)
    log_probs_taken = log_probs.gather(-1, tau.actions.unsqueeze(-1)).squeeze(-1)
    # alternatively with eindex
    # log_probs_taken = eindex(log_probs, tau.actions, "env time [env time] -> env time")
    return log_probs_taken


tests.test_compute_logprobs(compute_logprobs, PolicyNetwork)
```
</details>

## Building up to the loss function

We need to compute the probability ratio $\pi(a_t | s_t) / \pi_{old}(a_t | s_t)$ for each timestep taken in the rollout. This is used to compute the importance weights $\text{iw}_t$, which correct for the policy having changed since the rollout was collected. That lets us reuse a rollout for a few updates (`rollout_use_count > 1`), i.e. learn slightly off-policy. (Clamping the weights keeps this stable, but it's not PPO-style clipping, and the shipped config uses each rollout once with `use_iw=False`.) If `args.clip_coef` is not `None`, we also clamp the importance weights between `1 - args.clip_coef` and `1 + args.clip_coef`.

### Exercise - implement `compute_importance_weights`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10-15 minutes on this exercise.
> ```

Keep the result numerically stable by exponentiating the difference between the logprobs.
Gradients should **NOT** flow through the importance weights. Make sure to use `.detach()` to prevent this.

In [81]:
def compute_importance_weights(
    logprobs_taken: Float[Tensor, " num_envs num_steps"],
    tau: RolloutTensors,
    clip_coef: Optional[float]
) -> Float[Tensor, " num_envs num_steps"]:
    """
    Compute importance weights from log probabilities.

    Keeps the result numerically stable by exponentiating the difference between logprobs.
    Gradients should NOT flow through the importance weights (uses .detach()).
    Optionally clips the weights to [1 - clip_coef, 1 + clip_coef].
    """
    new_sampled_logprobs = logprobs_taken.detach()  # num_envs max_steps
    old_sampled_logprobs = tau.logprobs.detach()  # num_envs max_steps
    #prob_ratio = new_prob / old_prob
    #prob_ratio = e^new_logprob / e^old_logprob
    #prob_ratio = e^(new_logprob - old_logprob)
    prob_ratio = t.exp(new_sampled_logprobs - old_sampled_logprobs)
    if clip_coef is not None:
      return t.clamp(prob_ratio, min=(1 - clip_coef), max=(1 + clip_coef))
    else:
      return prob_ratio


tests.test_compute_importance_weights(compute_importance_weights)

All tests in `test_compute_importance_weights` passed!


<details><summary>Solution</summary>

```python
def compute_importance_weights(logprobs_taken, tau: RolloutTensors, clip_coef: Optional[float]) -> t.Tensor:
    """
    Compute importance weights from log probabilities.

    Keeps the result numerically stable by exponentiating the difference between logprobs.
    Gradients should NOT flow through the importance weights (uses .detach()).
    Optionally clips the weights to [1 - clip_coef, 1 + clip_coef].
    """
    iw = t.exp(logprobs_taken - tau.logprobs).detach()  # Detach to prevent gradient flow
    if clip_coef is not None:
        iw = t.clamp(iw, 1 - clip_coef, 1 + clip_coef)
    return iw


tests.test_compute_importance_weights(compute_importance_weights)
```
</details>

### Exercise - implement `normalize_returns`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 5 minutes on this exercise.
> ```

Normalize the returns by ensuring zero mean, unit variance **across all trajectories and timesteps**. Don't overthink this one, should be a one-liner.

In [86]:
def normalize_returns(returns: Float[Tensor, " num_envs num_steps"]) -> Float[Tensor, " num_envs num_steps"]:
    """
    Normalizes the returns by ensuring zero mean, unit variance across all trajectories and timesteps.
    """
    (std, mean) = t.std_mean(returns)
    return (returns - mean) / (std + 1e-8)


tests.test_normalize_returns(normalize_returns)

All tests in `test_normalize_returns` passed!


<details><summary>Solution</summary>

```python
def normalize_returns(returns: Float[Tensor, " num_envs num_steps"]) -> Float[Tensor, " num_envs num_steps"]:
    """
    Normalizes the returns by ensuring zero mean, unit variance across all trajectories and timesteps.
    """
    return (returns - returns.mean()) / (returns.std() + 1e-8)


tests.test_normalize_returns(normalize_returns)
```
</details>

### Exercise - implement `compute_reinforce_loss`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 5 minutes on this exercise.
> ```

This should be easy with everything else you've got.
The loss on timestep $t$ is
$$
\text{iw}_t \log \pi(a_t | s_t) \big( G_t - b_t \big)
$$
where $G_t$ is the return, $\text{iw}_t$ is the importance weight, and $\log \pi(a_t | s_t)$ are the logprobs, each for timestep $t$. You need to compute the baseline $b_t$ **inside this function**: use the mean return across the $N$ environments at each timestep, $b_t = \frac{1}{N} \sum_{i=1}^N G^{(i)}_t$, i.e. `returns.mean(dim=0, keepdim=True)`, and subtract it from `returns`. Remember to `.detach()` the result so no gradients flow through the advantage. (Separately, `compute_loss` may have already rescaled the returns to mean-zero unit-variance across *all* timesteps and trajectories, controlled by `args.normalize_returns` - that's variance reduction, not the baseline.) PPO uses a learned baseline called a critic, which we will see in the PPO section.

<details>
<summary>What kind of baseline is this?</summary>

Note that $b_t$ depends on the rollout index $t$, not on the state $s_t$: it's the average return of all the environments at the same position in the rollout. A baseline that doesn't depend on the action $a_t$ (or on anything that happens after it) leaves the policy gradient unbiased, and up to the subtlety below, this one qualifies. Because `full_reset=True` restarts every environment at the start of each rollout, for each environment's first episode the index $t$ is the time since the episode began, which is a crude but useful stand-in for the state. (It's *not* the state-value baseline $V(s_t)$ you'll meet in PPO.)

One subtlety: $b_t$ includes environment $i$'s own return $G^{(i)}_t$. A little algebra shows $G^{(i)}_t - b_t = \frac{N-1}{N} \big( G^{(i)}_t - \frac{1}{N-1} \sum_{j \neq i} G^{(j)}_t \big)$, i.e. it is exactly $\frac{N-1}{N}$ times the "leave-one-out" baseline, which is exactly unbiased (the other environments' returns are independent of environment $i$'s actions). So our gradient points the same way in expectation, just scaled down by $\frac{N-1}{N}$, which is like using a slightly smaller learning rate. That's negligible with 32 environments, but noticeable with the probes' 4.
</details>


The total loss is the mean of the losses over all timesteps, over all trajectories.

Note that despite the name, this "loss" is an objective we want to **maximize** (it's our estimate of the policy gradient objective, the "joy" $J(\theta)$): `VPGTrainer` creates its optimizer with `maximize=True`, so return it as written, without negating it.

A note on this convention: the policy gradient estimator is strictly a *sum* over timesteps within a trajectory, averaged over trajectories (environments). Taking the mean over timesteps as well just divides the loss by `num_steps`. Since every environment here runs for the same number of steps, this is only a constant rescaling of the gradient (equivalent to a different learning rate), so it doesn't change what the algorithm does. We use the mean over everything so that the loss scale doesn't depend on the rollout length.

In [96]:
def compute_reinforce_loss(
    returns: Float[Tensor, " num_envs num_steps"],
    logprobs_taken: Float[Tensor, " num_envs num_steps"],
    iw: Float[Tensor, " num_envs num_steps"],
) -> Float[Tensor, ""]:
    baseline = returns.detach().mean(dim=0, keepdim=True)  # 1 num_steps
    loss = t.mean(iw * logprobs_taken * (returns - baseline).detach())
    return loss


tests.test_compute_reinforce_loss(compute_reinforce_loss)

All tests in `test_compute_reinforce_loss` passed!


<details><summary>Solution</summary>

```python
def compute_reinforce_loss(
    returns: Float[Tensor, " num_envs num_steps"],
    logprobs_taken: Float[Tensor, " num_envs num_steps"],
    iw: Float[Tensor, " num_envs num_steps"],
) -> Float[Tensor, ""]:
    adv = returns - returns.mean(dim=0, keepdim=True)   # baseline per timestep across envs
    return (iw * logprobs_taken * adv.detach()).mean()


tests.test_compute_reinforce_loss(compute_reinforce_loss)
```
</details>

# 4️⃣ Training

> ##### Learning Objectives
>
> - Debug a policy gradient implementation with probe environments
> - Train an agent that balances the pole for the full 500 steps

## Live training visualisation (optional)

To *watch* training, `rl_utils.log_grid_video(obs, env.draw, dones=dones, live=...)` renders the first 16
environments of a rollout as a 4x4 grid (each environment draws its own cell via its `draw` method - this is
the same renderer used by DQN and PPO), encodes the frames as a single
autoplaying/looping MP4 (one ffmpeg encode, ~0.2s), logs it to wandb if a run is active, and displays it inline
if `live`. The training loop below calls it every `video_log_freq` gradient steps.

## Trainer

This is the function that will handle the full training loop. We've provided you with the template of a training loop which should be very similar to the DQN section's.

### Exercise - implement `VPGTrainer`

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 45 minutes on this exercise.
> ```

You should fill in the following methods. Ignore logging, can just copy from the solution later.

* `compute_loss` - this method should compute the loss for the VPG objective function.

The training loop is rather standard once everything else is done: we do a rollout, we cut the result into batches, compute the loss, and update the weights from each batch, so we've provided it for you.

In [98]:
class VPGTrainer:
    def __init__(self, args: VPGArgs):
        set_global_seeds(args.seed)
        self.args = args

        device = args.device

        self.run_name = f"{args.env_id}__{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        # Create our environments (any `env_id` registered in `ENVS`, see the setup cell)
        self.envs = make_envs(args.env_id, args.num_envs, args.seed, device)

        # Define some basic variables from our environment (note, we assume a single discrete action space)
        self.num_envs = args.num_envs
        self.action_shape = self.envs.action_space.shape
        self.num_actions = self.envs.action_space.n
        self.obs_shape = self.envs.observation_space.shape

        # Create our networks & optimizer
        self.policy_network = PolicyNetwork(self.obs_shape, self.num_actions).to(device)

        self.optimizer = t.optim.Adam(self.policy_network.parameters(), lr=args.lr, eps=1e-5, maximize=True)
        self.optimizer.zero_grad()

        # Create our agent
        self.agent = VPGAgent(envs=self.envs, policy_network=self.policy_network, args=self.args)

    def compute_loss(self, tau: RolloutTensors) -> tuple[t.Tensor, dict[str, Any]]:
        returns = compute_returns(tau.rewards, tau.dones, self.args.gamma)
        if self.args.normalize_returns:
            returns = normalize_returns(returns)
        logprobs_taken = compute_logprobs(tau, self.policy_network)
        iw = compute_importance_weights(logprobs_taken, tau, self.args.clip_coef)
        joy = compute_reinforce_loss(returns, logprobs_taken, iw)

        info = {
            "r_joy": joy.item(),
            "iw": iw.mean().item() if self.args.use_iw else None,
        }

        return joy, info

    def update_learning_rate(self, time_steps, args):
        if args.use_lr_decay and args.lr_frac > 0:
            progress = min(1.0, max(time_steps / args.total_timesteps, 0) / args.lr_frac)
            return (progress * args.lr_end) + ((1 - progress) * args.lr)
        return args.lr

    def train(self) -> None:
        """
        Trains the agent by generating rollouts and updating the policy.
        The progress bar tracks total environment steps.
        """
        if self.args.use_wandb:
            wandb.init(
                project=self.args.wandb_project_name,
                entity=self.args.wandb_entity,
                name=self.run_name,
            )
            wandb.watch(self.policy_network, log="all", log_freq=50)

        # --- Setup ---
        rollout = Rollout(max_steps=self.args.num_steps_per_rollout)

        # Calculate the total number of rollouts to perform
        env_steps_per_rollout = self.args.num_steps_per_rollout * self.args.num_envs
        num_updates = self.args.total_timesteps // env_steps_per_rollout
        train_steps = 0  # Counter for gradient updates
        next_video_at = 0  # gradient-step count at which the next grid video is due
        video_slot = LiveVideo(overwrite=self.args.overwrite_video)  # one display slot per run (see rl_utils.LiveVideo)

        # --- Training Loop ---
        # The progress bar is managed manually with a `with` statement.
        # `total` is set to the total environment steps we want to run.
        # The loop iterates `num_updates` times, not `total_timesteps` times.
        with tqdm(
            total=self.args.total_timesteps,
            unit=" env steps",
            unit_scale=True,
            desc="Training",
            miniters=1,
            mininterval=0.02,
        ) as pbar:
            env_steps_consumed = 0

            for update_num in range(num_updates):
                # 1. Generate a new rollout from the environment

                rollout = self.agent.gen_rollout(rollout)

                # 2. Split the rollout into batches along the num_envs dimension

                rollout_batches = rollout.get_batches(self.args.batch_size)

                # 3. Logging and Progress Bar Update
                # Undiscounted return of each env's *first* episode in this rollout
                tau = rollout.get()
                first_episode = (tau.dones.int().cumsum(dim=1) - tau.dones.int()) == 0  # no `done` strictly before this step
                episode_return = (tau.rewards * first_episode).sum(dim=1)  # (num_envs,)
                mean_return = episode_return.mean().item()
                # 95% confidence interval of average of return on first episode
                ci95_return = 1.96 * (episode_return.std() / self.args.num_envs**0.5).item()
                max_return = episode_return.max().item()

                # Log a 4x4 grid video of the rollout we already have (no extra env steps) every `video_log_freq`
                # gradient steps - to wandb if use_wandb, inline if live_viz - via the shared `rl_utils.log_grid_video`.
                # With rollout_use_count=1 and one batch per rollout that is one video every `video_log_freq` rollouts.
                want_video = self.args.video_log_freq and (self.args.use_wandb or self.args.live_viz)
                if want_video and train_steps >= next_video_at:
                    caption = f"rollout {rollout.timestep} steps | mean first-episode return {mean_return:.1f}"
                    log_grid_video(tau.obs, self.envs.draw, dones=tau.dones, step=env_steps_consumed, live=self.args.live_viz, slot=video_slot, caption=caption)
                    next_video_at += self.args.video_log_freq

                # 4. Advance env-step counter before gradient updates (one rollout collected)
                env_steps_consumed += self.args.num_steps_per_rollout * self.args.num_envs

                # 5. For each batch, perform multiple gradient updates
                for i in range(self.args.rollout_use_count):
                    for batch in rollout_batches:
                        loss, reinforce_info = self.compute_loss(batch)

                        info = reinforce_info

                        loss.backward()

                        self.optimizer.step()
                        self.optimizer.zero_grad()
                        train_steps += 1

                        new_lr = self.update_learning_rate(env_steps_consumed, self.args)

                        for pg in self.optimizer.param_groups:
                            pg["lr"] = new_lr

                        # Create info string to display in the progress bar
                        current_lr = self.optimizer.param_groups[0]["lr"]
                        info_dict = {
                            "joy": f"{info['r_joy']:.4f}",
                            "G": f"{mean_return:.2f} ± {ci95_return:.2f} (max: {max_return:.0f})",
                            "iw": f"{info['iw']:.4f}" if self.args.use_iw else None,
                            "lr": f"{current_lr:.2e}",
                        }

                        pbar.set_postfix(info_dict)

                # Progress bar advances once per rollout (env steps actually collected)
                pbar.update(self.args.num_steps_per_rollout * self.args.num_envs)

        # --- Cleanup ---
        self.envs.close()
        if self.args.use_wandb:
            wandb.finish()


tests.test_compute_loss(VPGTrainer, VPGArgs, Rollout)

All tests in `test_compute_loss` passed!


<details><summary>Solution</summary>

```python
class VPGTrainer:
    def __init__(self, args: VPGArgs):
        set_global_seeds(args.seed)
        self.args = args

        device = args.device

        self.run_name = f"{args.env_id}__{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        # Create our environments (any `env_id` registered in `ENVS`, see the setup cell)
        self.envs = make_envs(args.env_id, args.num_envs, args.seed, device)

        # Define some basic variables from our environment (note, we assume a single discrete action space)
        self.num_envs = args.num_envs
        self.action_shape = self.envs.action_space.shape
        self.num_actions = self.envs.action_space.n
        self.obs_shape = self.envs.observation_space.shape

        # Create our networks & optimizer
        self.policy_network = PolicyNetwork(self.obs_shape, self.num_actions).to(device)

        self.optimizer = t.optim.Adam(self.policy_network.parameters(), lr=args.lr, eps=1e-5, maximize=True)
        self.optimizer.zero_grad()

        # Create our agent
        self.agent = VPGAgent(envs=self.envs, policy_network=self.policy_network, args=self.args)

    def compute_loss(self, tau: RolloutTensors) -> tuple[t.Tensor, dict[str, Any]]:
        returns = compute_returns(tau.rewards, tau.dones, self.args.gamma)  # (num_envs, timestep)

        if self.args.normalize_returns:
            returns = normalize_returns(returns)

        logprobs_taken = compute_logprobs(tau, self.policy_network)

        iw = compute_importance_weights(logprobs_taken, tau, self.args.clip_coef) if self.args.use_iw else t.ones_like(logprobs_taken)
        joy = compute_reinforce_loss(returns, logprobs_taken, iw)


        info = {
            "r_joy": joy.item(),
            "iw": iw.mean().item() if self.args.use_iw else None,
        }

        return joy, info

    def update_learning_rate(self, time_steps, args):
        if args.use_lr_decay and args.lr_frac > 0:
            progress = min(1.0, max(time_steps / args.total_timesteps, 0) / args.lr_frac)
            return (progress * args.lr_end) + ((1 - progress) * args.lr)
        return args.lr

    def train(self) -> None:
        """
        Trains the agent by generating rollouts and updating the policy.
        The progress bar tracks total environment steps.
        """
        if self.args.use_wandb:
            wandb.init(
                project=self.args.wandb_project_name,
                entity=self.args.wandb_entity,
                name=self.run_name,
            )
            wandb.watch(self.policy_network, log="all", log_freq=50)

        # --- Setup ---
        rollout = Rollout(max_steps=self.args.num_steps_per_rollout)

        # Calculate the total number of rollouts to perform
        env_steps_per_rollout = self.args.num_steps_per_rollout * self.args.num_envs
        num_updates = self.args.total_timesteps // env_steps_per_rollout
        train_steps = 0  # Counter for gradient updates
        next_video_at = 0  # gradient-step count at which the next grid video is due
        video_slot = LiveVideo(overwrite=self.args.overwrite_video)  # one display slot per run (see rl_utils.LiveVideo)

        # --- Training Loop ---
        # The progress bar is managed manually with a `with` statement.
        # `total` is set to the total environment steps we want to run.
        # The loop iterates `num_updates` times, not `total_timesteps` times.
        with tqdm(
            total=self.args.total_timesteps,
            unit=" env steps",
            unit_scale=True,
            desc="Training",
            miniters=1,
            mininterval=0.02,
        ) as pbar:
            env_steps_consumed = 0

            for update_num in range(num_updates):
                # 1. Generate a new rollout from the environment

                rollout = self.agent.gen_rollout(rollout)

                # 2. Split the rollout into batches along the num_envs dimension

                rollout_batches = rollout.get_batches(self.args.batch_size)

                # 3. Logging and Progress Bar Update
                # Undiscounted return of each env's *first* episode in this rollout
                tau = rollout.get()
                first_episode = (tau.dones.int().cumsum(dim=1) - tau.dones.int()) == 0  # no `done` strictly before this step
                episode_return = (tau.rewards * first_episode).sum(dim=1)  # (num_envs,)
                mean_return = episode_return.mean().item()
                # 95% confidence interval of average of return on first episode
                ci95_return = 1.96 * (episode_return.std() / self.args.num_envs**0.5).item()
                max_return = episode_return.max().item()

                # Log a 4x4 grid video of the rollout we already have (no extra env steps) every `video_log_freq`
                # gradient steps - to wandb if use_wandb, inline if live_viz - via the shared `rl_utils.log_grid_video`.
                # With rollout_use_count=1 and one batch per rollout that is one video every `video_log_freq` rollouts.
                want_video = self.args.video_log_freq and (self.args.use_wandb or self.args.live_viz)
                if want_video and train_steps >= next_video_at:
                    caption = f"rollout {rollout.timestep} steps | mean first-episode return {mean_return:.1f}"
                    log_grid_video(tau.obs, self.envs.draw, dones=tau.dones, step=env_steps_consumed, live=self.args.live_viz, slot=video_slot, caption=caption)
                    next_video_at += self.args.video_log_freq

                # 4. Advance env-step counter before gradient updates (one rollout collected)
                env_steps_consumed += self.args.num_steps_per_rollout * self.args.num_envs

                # 5. For each batch, perform multiple gradient updates
                for i in range(self.args.rollout_use_count):
                    for batch in rollout_batches:
                        loss, reinforce_info = self.compute_loss(batch)

                        info = reinforce_info

                        loss.backward()

                        self.optimizer.step()
                        self.optimizer.zero_grad()
                        train_steps += 1

                        new_lr = self.update_learning_rate(env_steps_consumed, self.args)

                        for pg in self.optimizer.param_groups:
                            pg["lr"] = new_lr

                        # Create info string to display in the progress bar
                        current_lr = self.optimizer.param_groups[0]["lr"]
                        info_dict = {
                            "joy": f"{info['r_joy']:.4f}",
                            "G": f"{mean_return:.2f} ± {ci95_return:.2f} (max: {max_return:.0f})",
                            "iw": f"{info['iw']:.4f}" if self.args.use_iw else None,
                            "lr": f"{current_lr:.2e}",
                        }

                        pbar.set_postfix(info_dict)

                # Progress bar advances once per rollout (env steps actually collected)
                pbar.update(self.args.num_steps_per_rollout * self.args.num_envs)

        # --- Cleanup ---
        self.envs.close()
        if self.args.use_wandb:
            wandb.finish()


tests.test_compute_loss(VPGTrainer, VPGArgs, Rollout)
```
</details>

## Probes

As in the DQN section, we will be using probes to test our model. They've been implemented for you.

<details>
<summary> What's with the <code>t.set_num_threads</code> line in the setup code? </summary>

Note that if you are training on the CPU (which for a network this small
can often be faster than the GPU!), having too many threads can slow training
as the overhead between splitting the work over many threads can hurt.
As such, we've added the line

```python
t.set_num_threads(min(4, t.get_num_threads()))
```

to the setup code, which caps the number of CPU threads PyTorch uses at 4 (and does nothing if
you have fewer). Feel free to adjust it or remove it entirely.
On a beefy machine with dozens of threads, I found a x10 slowdown if
the number of threads were not capped.

</details>

In [99]:
def test_probe(probe_idx: int):
    """
    Tests a probe environment by training a policy on it & verifying that the action probabilities are close to the
    optimal ones. Only probes 4 and 5 apply: probes 1-3 have a single action, so they only test value learning, and VPG
    has no value function.
    """
    assert probe_idx in (4, 5), "VPG has no value function, so only probes 4 and 5 (which test the policy) apply."
    # Train our network
    args = VPGArgs(
        env_id=f"Probe{probe_idx}-v0",
        wandb_project_name=f"test-probe-{probe_idx}",
        total_timesteps=[None, None, None, 2500, 5000][probe_idx - 1],
        num_steps_per_rollout=16,  # one-step episodes: short rollouts = more (cheap) updates per env step
        lr=5e-3,
        num_envs=4,
        video_log_freq=None,
        use_wandb=False,
        device="cpu",
        clip_coef=None,
        normalize_returns=False,
        rollout_use_count=1,
    )
    trainer = VPGTrainer(args)
    trainer.train()
    agent = trainer.agent

    # Get the correct set of observations, and corresponding values we expect
    obs_for_probes = [[[0.0]], [[-1.0], [+1.0]], [[0.0], [1.0]], [[0.0]], [[0.0], [1.0]]]
    expected_probs_for_probes = [None, None, None, [[0.0, 1.0]], [[1.0, 0.0], [0.0, 1.0]]]
    tolerances = [1e-3, 1e-3, 1e-3, 2e-3, 2e-3]
    obs = t.tensor(obs_for_probes[probe_idx - 1]).to(args.device)

    # Calculate the actual value & probs, and verify them
    with t.inference_mode():
        probs = agent.policy_network(obs).softmax(-1)
    expected_probs = expected_probs_for_probes[probe_idx - 1]
    if expected_probs is not None:
        print(f"Probs: {probs}")
        print(f"Expected probs: {t.tensor(expected_probs).to(args.device)}")
        # A bf16 network (e.g. under autocast) can't represent these targets to better than its eps
        atol = max(tolerances[probe_idx - 1], t.finfo(probs.dtype).eps)
        t.testing.assert_close(probs.float(), t.tensor(expected_probs).to(args.device), atol=atol, rtol=0)
    print(f"Probe {probe_idx} tests passed!\n")

for probe_idx in [4, 5]:
    test_probe(probe_idx)

Training:   0%|          | 0.00/2.50k [00:00<?, ? env steps/s]

Probs: tensor([[1.4115e-06, 1.0000e+00]])
Expected probs: tensor([[0., 1.]])
Probe 4 tests passed!



Training:   0%|          | 0.00/5.00k [00:00<?, ? env steps/s]

Probs: tensor([[9.9987e-01, 1.3019e-04],
        [3.6439e-06, 1.0000e+00]])
Expected probs: tensor([[1., 0.],
        [0., 1.]])
Probe 5 tests passed!



## Training Run

Vanilla Policy Gradient can often be a bit finicky and unstable to train (which is why in practice we use PPO instead).
None-the-less, I've tried to find a good set of hyperparameters such that it trains to an optimal policy in around 10 seconds, on a CPU or a GPU. (This relies on the `t.set_num_threads` line in the setup cell if you're training on a CPU with too many cores: PyTorch's default of one thread per core is counterproductive for a network this small.)
By default (`live_viz=True`) a 4x4 grid video of the agent is rendered in the notebook every `video_log_freq` gradient steps as it trains; pass `live_viz=False` to turn that off (videos are only sent to wandb if `use_wandb=True`). The background of a cell flashes pink when that environment's agent dies and is reset, so you can easily see when a new episode starts.

In [100]:
args_fast = VPGArgs(
    use_wandb=False,
    num_envs=32,
    num_batches_per_rollout=1,
    total_timesteps=640_000,
    num_steps_per_rollout=500,
    rollout_use_count=1,  # this seems to matter a lot
    normalize_returns=True,
    lr=2e-2,  # very high, but the per-timestep baseline across envs keeps the gradient low-variance enough to take it
    use_lr_decay=False,
    use_iw=False,  # don't need it if we only use each rollout once
    gamma=0.99,
    seed=1337,
    device=device,
    video_log_freq=10,
)
trainer = VPGTrainer(args_fast)
trainer.train()

Training:   0%|          | 0.00/640k [00:00<?, ? env steps/s]